# DINOv3 crop linear probe

使用 `data/dataset/metadata.csv` 与已存盘 crop 图像快速验证：DINOv3 冻结特征 + 线性分类头在 train/val/test 三分割上的可靠性。

特点：
- 直接读取 stage 14 生成的 `data/dataset/images/*`，不再从原图 bbox 在线裁剪。
- 复用 pipeline 配置中的 DINOv3 模型名、batch size、学习率、weight decay，并复用 stage 10 的 `LinearHead`。
- 学习率保持固定，不使用 cyclic/sawtooth scheduler。
- 用 validation loss 早停，最后在 test split 上评估。

In [ ]:
from pathlib import Path
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import AutoImageProcessor, AutoModel
from huggingface_hub import login

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for path in candidates:
        if (path / "pyproject.toml").exists() and (path / "pipeline").exists():
            return path
    raise RuntimeError(f"Cannot find wakareeru project root from {start}")


PROJECT_ROOT = find_project_root()
for path in [PROJECT_ROOT, PROJECT_ROOT / "pipeline"]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

import utils
from stage_09_DINOv3_feature_extraction import get_torch_device, FeatureCollator
from stage_10_train_loss_tracking import LinearHead

tqdm.pandas()

config = utils.load_pipeline_config(PROJECT_ROOT / "config" / "pipeline_config.yaml")
data_root = utils.get_data_root(config)
dataset_root = utils.join_data_root(config["path"]["dataset_dir"], config=config)
metadata_path = dataset_root / "metadata.csv"
labels_path = dataset_root / "labels.csv"

print(f"project_root={PROJECT_ROOT}")
print(f"dataset_root={dataset_root}")

In [ ]:
# 实验参数。先保持小而直接，便于快速判断数据可靠性。
SEED = 42
LABEL_COLUMN = "label"
MIN_SAMPLES_PER_CLASS = 3
VAL_RATIO = 0.15
TEST_RATIO = 0.15

FEATURE_BATCH_SIZE = int(config["noise_detection"].get("feature_extraction_batch_size", 16))
HEAD_BATCH_SIZE = int(config["noise_detection"].get("linear_head_train_batch_size", 32))
# Notebook 中自定义 Dataset 定义在 __main__，macOS/Python 3.12 的 spawn 多进程
# 无法稳定 pickle；这里固定为 0，避免 DataLoader worker 导入失败。
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10
MIN_DELTA = 1e-4

LR = float(config["loss_noise_tracking"].get("learning_rate_high", 1e-3))
WEIGHT_DECAY = float(config["loss_noise_tracking"].get("weight_decay", 1e-4))
HF_MODEL_NAME = config["noise_detection"]["hf_model_name"]

feature_cache_dir = utils.join_data_root(
    config["noise_detection"].get("feature_cache_dir", "feature_cache"),
    config=config,
)
feature_cache_path = feature_cache_dir / "dataset_crop_dinov3_features.pt"

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

print(HF_MODEL_NAME, FEATURE_BATCH_SIZE, HEAD_BATCH_SIZE, LR, WEIGHT_DECAY)

## 读取 metadata 并做三分割

`MIN_SAMPLES_PER_CLASS=3` 是为了让每个类别至少能落入 train/val/test。样本太少的类先过滤掉，否则 stratified split 本身不可靠。

In [ ]:
metadata = pd.read_csv(metadata_path)
metadata["abs_image_path"] = metadata["image_path"].map(lambda p: dataset_root / str(p))
metadata["crop_id"] = metadata["image_path"].str.extract(r"_(\d+)\.").astype(int)
metadata[LABEL_COLUMN] = metadata[LABEL_COLUMN].astype(str)

exists_mask = metadata["abs_image_path"].progress_map(Path.exists)
if not exists_mask.all():
    print(f"missing image files: {(~exists_mask).sum()}")
metadata = metadata.loc[exists_mask].copy().reset_index(drop=True)

class_counts = metadata[LABEL_COLUMN].value_counts()
kept_labels = class_counts[class_counts >= MIN_SAMPLES_PER_CLASS].index
df = metadata[metadata[LABEL_COLUMN].isin(kept_labels)].copy().reset_index(drop=True)

labels = sorted(df[LABEL_COLUMN].unique())
label_to_id = {label: idx for idx, label in enumerate(labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
df["label_id_exp"] = df[LABEL_COLUMN].map(label_to_id).astype(int)

train_val_df, test_df = train_test_split(
    df,
    test_size=TEST_RATIO,
    random_state=SEED,
    stratify=df["label_id_exp"],
)
relative_val_ratio = VAL_RATIO / (1.0 - TEST_RATIO)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=relative_val_ratio,
    random_state=SEED,
    stratify=train_val_df["label_id_exp"],
)

split_frames = {
    "train": train_df.reset_index(drop=True),
    "val": val_df.reset_index(drop=True),
    "test": test_df.reset_index(drop=True),
}
for split, part in tqdm(split_frames.items(), desc="split summary"):
    print(split, part.shape, "classes=", part[LABEL_COLUMN].nunique())

display(df[LABEL_COLUMN].value_counts().rename("count").to_frame().head(30))

## DINOv3 特征抽取

如果 `feature_cache_path` 已存在，默认直接加载；想重抽特征时把 `REBUILD_FEATURE_CACHE=True`。

In [ ]:
class StoredCropDataset(torch.utils.data.Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row["abs_image_path"]).convert("RGB")
        meta = {
            "crop_id": int(row["crop_id"]),
            "label": str(row[LABEL_COLUMN]),
            "split": row["split"],
        }
        return image, meta


def extract_features(frame: pd.DataFrame, processor, model, label_to_id: dict[str, int], batch_size: int):
    dataset = StoredCropDataset(frame)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        collate_fn=FeatureCollator(processor, label_to_id),
    )
    features, labels_out, crop_ids = [], [], []
    model.eval()
    with torch.inference_mode():
        for image_tensors, labels_cpu, crop_ids_cpu in tqdm(loader, desc="DINOv3 features"):
            image_tensors = image_tensors.to(model.device)
            outputs = model(image_tensors)
            features.append(outputs.pooler_output.detach().cpu())
            labels_out.append(labels_cpu.cpu())
            crop_ids.append(crop_ids_cpu.cpu())
    return {
        "features": torch.cat(features, dim=0),
        "labels": torch.cat(labels_out, dim=0),
        "crop_ids": torch.cat(crop_ids, dim=0),
    }

In [ ]:
REBUILD_FEATURE_CACHE = False

split_df = pd.concat(
    [part.assign(split=split) for split, part in split_frames.items()],
    ignore_index=True,
)

if feature_cache_path.exists() and not REBUILD_FEATURE_CACHE:
    feature_cache = torch.load(feature_cache_path, map_location="cpu")
    print(f"loaded feature cache: {feature_cache_path}")
else:
    token = os.getenv("HF_TOKEN")
    if token:
        login(token=token)
    device = get_torch_device()
    processor = AutoImageProcessor.from_pretrained(HF_MODEL_NAME)
    model = AutoModel.from_pretrained(HF_MODEL_NAME, device_map="auto")
    feature_parts = []
    for split in tqdm(["train", "val", "test"], desc="feature splits"):
        part = split_df[split_df["split"] == split].reset_index(drop=True)
        extracted = extract_features(part, processor, model, label_to_id, FEATURE_BATCH_SIZE)
        extracted["splits"] = [split] * len(part)
        feature_parts.append(extracted)
    feature_cache = {
        "features": torch.cat([p["features"] for p in feature_parts], dim=0),
        "labels": torch.cat([p["labels"] for p in feature_parts], dim=0),
        "crop_ids": torch.cat([p["crop_ids"] for p in feature_parts], dim=0),
        "splits": sum([p["splits"] for p in feature_parts], []),
        "label_to_id": label_to_id,
        "id_to_label": id_to_label,
        "model_name": HF_MODEL_NAME,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    feature_cache_dir.mkdir(parents=True, exist_ok=True)
    torch.save(feature_cache, feature_cache_path)
    print(f"saved feature cache: {feature_cache_path}")

print(feature_cache["features"].shape, len(feature_cache["label_to_id"]))

## 固定学习率训练线性头 + 早停

In [ ]:
class FeatureSplitDataset(torch.utils.data.Dataset):
    def __init__(self, feature_cache: dict, split: str):
        split_mask = np.array(feature_cache["splits"]) == split
        self.features = feature_cache["features"][split_mask].float()
        self.labels = feature_cache["labels"][split_mask].long()
        self.crop_ids = feature_cache["crop_ids"][split_mask].long()

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx], self.crop_ids[idx]


def evaluate_head(head, loader, criterion, device, desc="evaluate"):
    head.eval()
    losses, preds, labels, crop_ids = [], [], [], []
    with torch.inference_mode():
        for x_cpu, y_cpu, crop_ids_cpu in tqdm(loader, desc=desc, leave=False):
            x = x_cpu.to(device)
            y = y_cpu.to(device)
            logits = head(x)
            loss = criterion(logits, y)
            pred = logits.argmax(dim=1)
            losses.append(loss.detach().cpu().repeat(len(y_cpu)))
            preds.append(pred.detach().cpu())
            labels.append(y_cpu.cpu())
            crop_ids.append(crop_ids_cpu.cpu())
    preds = torch.cat(preds).numpy()
    labels = torch.cat(labels).numpy()
    crop_ids = torch.cat(crop_ids).numpy()
    return {
        "loss": float(torch.cat(losses).mean().item()),
        "accuracy": float(accuracy_score(labels, preds)),
        "preds": preds,
        "labels": labels,
        "crop_ids": crop_ids,
    }


train_ds = FeatureSplitDataset(feature_cache, "train")
val_ds = FeatureSplitDataset(feature_cache, "val")
test_ds = FeatureSplitDataset(feature_cache, "test")

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=HEAD_BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=HEAD_BATCH_SIZE, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=HEAD_BATCH_SIZE, shuffle=False)

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
head = LinearHead(
    input_dim=int(feature_cache["features"].shape[1]),
    num_classes=len(feature_cache["label_to_id"]),
).to(device)
optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = torch.nn.CrossEntropyLoss()

best_val_loss = float("inf")
best_state = None
bad_epochs = 0
history = []

for epoch in tqdm(range(MAX_EPOCHS), desc="linear head"):
    head.train()
    running_loss = 0.0
    running_correct = 0
    running_n = 0
    for x_cpu, y_cpu, _ in tqdm(train_loader, desc=f"epoch {epoch} train", leave=False):
        x = x_cpu.to(device)
        y = y_cpu.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = head(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += float(loss.detach().cpu()) * int(y.numel())
        running_correct += int(logits.argmax(dim=1).eq(y).sum().detach().cpu())
        running_n += int(y.numel())

    train_loss = running_loss / max(1, running_n)
    train_acc = running_correct / max(1, running_n)
    val_metrics = evaluate_head(head, val_loader, criterion, device, desc=f"epoch {epoch} val")
    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)
    print(row)

    if val_metrics["loss"] < best_val_loss - MIN_DELTA:
        best_val_loss = val_metrics["loss"]
        best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"early stopping at epoch={epoch}, best_val_loss={best_val_loss:.4f}")
            break

history_df = pd.DataFrame(history)
if best_state is not None:
    head.load_state_dict(best_state)
display(history_df.tail())

## Test 评估与错误样本查看

In [ ]:
test_metrics = evaluate_head(head, test_loader, criterion, device, desc="test")
print({k: v for k, v in test_metrics.items() if k in ["loss", "accuracy"]})

target_names = [feature_cache["id_to_label"][i] for i in range(len(feature_cache["id_to_label"]))]
print(classification_report(
    test_metrics["labels"],
    test_metrics["preds"],
    labels=list(range(len(target_names))),
    target_names=target_names,
    zero_division=0,
))

In [ ]:
result_df = pd.DataFrame({
    "crop_id": test_metrics["crop_ids"],
    "label_id": test_metrics["labels"],
    "pred_id": test_metrics["preds"],
})
result_df["label"] = result_df["label_id"].map(feature_cache["id_to_label"])
result_df["pred"] = result_df["pred_id"].map(feature_cache["id_to_label"])
result_df["correct"] = result_df["label_id"] == result_df["pred_id"]

test_meta = split_df[split_df["split"] == "test"][["crop_id", "image_path", LABEL_COLUMN, "series", "fine_grained_series", "submodel"]]
result_df = result_df.merge(test_meta, on="crop_id", how="left")
display(result_df[~result_df["correct"]].head(50))

In [ ]:
import matplotlib.pyplot as plt

history_df[["train_loss", "val_loss"]].plot(title="loss")
plt.show()
history_df[["train_accuracy", "val_accuracy"]].plot(title="accuracy")
plt.show()